# Final Exam Workflow

Integrated notebook for the exam-oriented NQS workflow. It keeps orchestration in the notebook and delegates reusable experiment logic to `demo_helper.py` and reusable export/plotting logic to `final_exam_helper.py`.


In [ ]:
from pathlib import Path

import pandas as pd

from demos.demo_helper import (
    build_system,
    exact_observables_summary,
    run_architecture_disorder_comparison,
    run_ghz_bonus_workflow,
    run_hamiltonian_system_size_sweep,
    run_vmc_experiment,
)
from demos.final_exam_helper import (
    build_output_manifest,
    ensure_report_output_dir,
    plot_architecture_summary,
    plot_energy_benchmark,
    plot_entropy_scan,
    plot_training_history,
    save_report_figure,
    save_report_table,
)


In [ ]:
output_dir = ensure_report_output_dir("final_exam_notebook")
manifest_entries = []

architecture_configs = {
    "RBM": {"alpha": 1},
    "FFNN": {"hidden_dims": (16, 8)},
    "CNN": {"channels": (4, 2), "kernel_size": (1, 1)},
}

critical_sweep_points = [
    {"label": "tfim_critical", "hamiltonian": "tfim", "lattice_shape": (2, 2), "h": 1.0},
    {"label": "tfim_away", "hamiltonian": "tfim", "lattice_shape": (2, 2), "h": 0.5},
    {"label": "j1j2_frustrated", "hamiltonian": "j1_j2", "lattice_shape": (2, 2), "J1": 1.0, "J2": 0.5},
]

larger_system_points = [
    {"label": "tfim_3x3", "hamiltonian": "tfim", "lattice_shape": (3, 3), "h": 1.0},
    {"label": "j1j2_3x3", "hamiltonian": "j1_j2", "lattice_shape": (3, 3), "J1": 1.0, "J2": 0.4},
]


## 1. ED Benchmarks And Small-System Entropy


In [ ]:
ed_system = build_system(lattice_shape=(2, 2), hamiltonian="tfim", h=1.0)
ed_exact = exact_observables_summary(ed_system["operator"])
ed_vmc = run_vmc_experiment(
    model_name="RBM",
    model_kwargs={"alpha": 1},
    lattice_shape=(2, 2),
    hamiltonian="tfim",
    h=1.0,
    n_iter=8,
    callback_every=1,
)
ed_summary = pd.DataFrame(
    [
        {
            "sweep_label": "tfim_2x2",
            "final_energy": ed_vmc["final_energy"],
            "exact_ground_energy": ed_vmc["exact"]["ground_energy"],
        }
    ]
)
energy_figure = plot_energy_benchmark(ed_summary, title="TFIM 2x2 Energy Benchmark")
entropy_figure = plot_entropy_scan(
    ed_exact["entropy_table"].assign(model="Exact"),
    line_column="model",
    title="Exact Renyi-2 Entropy Scan",
)
ed_table_paths = save_report_table(ed_exact["entropy_table"], "ed_entropy_scan", output_dir)
ed_energy_path = save_report_figure(energy_figure, "ed_energy_benchmark", output_dir)
ed_entropy_path = save_report_figure(entropy_figure, "ed_entropy_scan", output_dir)
manifest_entries.extend(
    [
        {"section": "ed", "name": "entropy_table_csv", "path": str(ed_table_paths["csv"])},
        {"section": "ed", "name": "entropy_table_html", "path": str(ed_table_paths["html"])},
        {"section": "ed", "name": "energy_figure", "path": str(ed_energy_path)},
        {"section": "ed", "name": "entropy_figure", "path": str(ed_entropy_path)},
    ]
)
ed_exact["entropy_table"]


## 2. Architecture Comparisons And Entropy Scans


In [ ]:
architecture_result = run_architecture_disorder_comparison(
    architecture_configs=architecture_configs,
    seeds=(0, 1, 2),
    lattice_shape=(2, 2),
    hamiltonian="tfim",
    entropy_n_independent_runs=1,
)
architecture_figure = plot_architecture_summary(architecture_result["summary_table"])
architecture_entropy_figure = plot_entropy_scan(
    architecture_result["entropy_scan_table"],
    line_column="model",
    title="Architecture Renyi-2 Scan",
)
architecture_summary_paths = save_report_table(
    architecture_result["summary_table"],
    "architecture_summary",
    output_dir,
)
architecture_entropy_paths = save_report_table(
    architecture_result["entropy_scan_table"],
    "architecture_entropy_scan",
    output_dir,
)
architecture_figure_path = save_report_figure(architecture_figure, "architecture_summary", output_dir)
architecture_entropy_figure_path = save_report_figure(
    architecture_entropy_figure,
    "architecture_entropy_scan",
    output_dir,
)
manifest_entries.extend(
    [
        {"section": "architecture", "name": "summary_csv", "path": str(architecture_summary_paths["csv"])},
        {"section": "architecture", "name": "summary_html", "path": str(architecture_summary_paths["html"])},
        {"section": "architecture", "name": "entropy_csv", "path": str(architecture_entropy_paths["csv"])},
        {"section": "architecture", "name": "entropy_html", "path": str(architecture_entropy_paths["html"])},
        {"section": "architecture", "name": "summary_figure", "path": str(architecture_figure_path)},
        {"section": "architecture", "name": "entropy_figure", "path": str(architecture_entropy_figure_path)},
    ]
)
architecture_result["summary_table"]


## 3. Training-Time Renyi-2 For Critical And Noncritical Sweeps


In [ ]:
sweep_result = run_hamiltonian_system_size_sweep(
    sweep_points=critical_sweep_points,
    model_name="RBM",
    model_kwargs={"alpha": 1},
    n_iter=10,
    callback_every=1,
)
training_energy_figure = plot_training_history(
    sweep_result["training_history_table"],
    "energy",
    title="Training Energy During Sweep",
)
training_entropy_figure = plot_training_history(
    sweep_result["training_history_table"],
    "renyi2_entropy",
    title="Training Renyi-2 During Sweep",
)
sweep_summary_paths = save_report_table(sweep_result["summary_table"], "training_sweep_summary", output_dir)
sweep_history_paths = save_report_table(
    sweep_result["training_history_table"],
    "training_sweep_history",
    output_dir,
)
training_energy_path = save_report_figure(training_energy_figure, "training_energy", output_dir)
training_entropy_path = save_report_figure(training_entropy_figure, "training_entropy", output_dir)
manifest_entries.extend(
    [
        {"section": "training", "name": "summary_csv", "path": str(sweep_summary_paths["csv"])},
        {"section": "training", "name": "summary_html", "path": str(sweep_summary_paths["html"])},
        {"section": "training", "name": "history_csv", "path": str(sweep_history_paths["csv"])},
        {"section": "training", "name": "history_html", "path": str(sweep_history_paths["html"])},
        {"section": "training", "name": "energy_figure", "path": str(training_energy_path)},
        {"section": "training", "name": "entropy_figure", "path": str(training_entropy_path)},
    ]
)
sweep_result["summary_table"]


## 4. Larger-System Studies


In [ ]:
larger_system_result = run_hamiltonian_system_size_sweep(
    sweep_points=larger_system_points,
    model_name="RBM",
    model_kwargs={"alpha": 1},
    n_iter=6,
    callback_every=1,
)
larger_system_entropy_figure = plot_training_history(
    larger_system_result["training_history_table"],
    "renyi2_entropy",
    title="Larger-System Renyi-2 During Training",
)
larger_system_paths = save_report_table(larger_system_result["summary_table"], "larger_system_summary", output_dir)
larger_system_figure_path = save_report_figure(
    larger_system_entropy_figure,
    "larger_system_entropy",
    output_dir,
)
manifest_entries.extend(
    [
        {"section": "larger_systems", "name": "summary_csv", "path": str(larger_system_paths["csv"])},
        {"section": "larger_systems", "name": "summary_html", "path": str(larger_system_paths["html"])},
        {"section": "larger_systems", "name": "entropy_figure", "path": str(larger_system_figure_path)},
    ]
)
larger_system_result["summary_table"]


## 5. GHZ Bonus Workflow


In [ ]:
ghz_result = run_ghz_bonus_workflow(
    model_name="RBM",
    model_kwargs={"alpha": 1},
    lattice_shape=(2, 2),
    n_iter=8,
    callback_every=1,
)
ghz_metrics_table = pd.DataFrame([ghz_result["ghz_metrics"]])
ghz_paths = save_report_table(ghz_metrics_table, "ghz_metrics", output_dir)
manifest_entries.extend(
    [
        {"section": "ghz", "name": "metrics_csv", "path": str(ghz_paths["csv"])},
        {"section": "ghz", "name": "metrics_html", "path": str(ghz_paths["html"])},
    ]
)
ghz_metrics_table


## 6. Output Manifest


In [ ]:
manifest_table = build_output_manifest(manifest_entries)
manifest_table
